# 05 — WAHIS and Climate Data Integration

## Purpose

This notebook integrates the validated WAHIS poultry disease surveillance
analytical table with semester-level climate data at the Nigerian ADM1
administrative level.

The integration is designed around a reproducible spatial-temporal key:

**ADM1 + Year + Semester**

### Data sources

1. WAHIS analytical table
   - `data/processed/wahis_analytical_table.csv`

2. Validated WAHIS administrative boundary lookup
   - `data/processed/wahis_administrative_boundary_lookup.csv`

3. Semester-level climate dataset
   - `data/interim/climate/adm1_semester_climate_2019_2025.csv`

### Important methodological principles

- Missing WAHIS outbreak counts are treated as unavailable, not zero.
- Administrative geography is reconciled using the validated boundary lookup.
- No automatic fuzzy geographic matching is performed.
- WAHIS semester values such as `Jan-Jun 2019` are standardised to
  `Jan-Jun` while the year is retained separately.
- Climate data are integrated only where ADM1, year, and semester keys match.
- The integration must preserve the original WAHIS records.
- Duplicate climate observations at the ADM1-year-semester level must not
  be introduced by the merge.

### Expected output

The final integrated dataset will be saved as:

`data/processed/wahis_climate_integrated.csv`

In [3]:
# ============================================================
# PROJECT PATH SETUP
# Notebook: 05_Data_Integration_v2.ipynb
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# Determine the project root.
#
# This notebook is stored inside the project's `notebooks/`
# directory, so the project root is one level above the
# current working directory.
# ------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent


# ------------------------------------------------------------
# Define the main data directories.
# ------------------------------------------------------------
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"


# ------------------------------------------------------------
# Define the input files used in this notebook.
# ------------------------------------------------------------
WAHIS_FILE = (
    PROCESSED_DIR /
    "wahis_analytical_table.csv"
)

BOUNDARY_FILE = (
    PROCESSED_DIR /
    "wahis_administrative_boundary_lookup.csv"
)

CLIMATE_FILE = (
    INTERIM_DIR /
    "climate" /
    "adm1_semester_climate_2019_2025.csv"
)


# ------------------------------------------------------------
# Define the output file.
# ------------------------------------------------------------
OUTPUT_FILE = (
    PROCESSED_DIR /
    "wahis_climate_integrated.csv"
)


# ------------------------------------------------------------
# Display and validate project paths.
# ------------------------------------------------------------
print("=" * 70)
print("PROJECT PATH SETUP")
print("=" * 70)

print(f"\nProject root:")
print(PROJECT_ROOT)

print(f"\nProcessed data directory:")
print(PROCESSED_DIR)

print(f"\nWAHIS analytical table:")
print(WAHIS_FILE)

print(f"\nBoundary lookup:")
print(BOUNDARY_FILE)

print(f"\nSemester climate dataset:")
print(CLIMATE_FILE)

print(f"\nOutput file:")
print(OUTPUT_FILE)


# ------------------------------------------------------------
# Confirm that all required input files exist.
# ------------------------------------------------------------
required_files = {
    "WAHIS analytical table": WAHIS_FILE,
    "Boundary lookup": BOUNDARY_FILE,
    "Semester climate dataset": CLIMATE_FILE,
}

print("\nInput file validation:")
for name, path in required_files.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"  {status}: {name}")


missing_files = [
    name
    for name, path in required_files.items()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following required input files are missing: "
        + ", ".join(missing_files)
    )

print("\nProject path validation completed successfully.")

PROJECT PATH SETUP

Project root:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria

Processed data directory:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\data\processed

WAHIS analytical table:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\data\processed\wahis_analytical_table.csv

Boundary lookup:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\data\processed\wahis_administrative_boundary_lookup.csv

Semester climate dataset:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\data\interim\climate\adm1_semester_climate_2019_2025.csv

Output file:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\data\processed\wahis_climate_integrated.csv

Input file validation:
  FOUND: WAHIS analytical table
  FOUND: Boundary lookup
  FOUND: Semester climate dataset

Project path validation completed successfully.


In [4]:
# ============================================================
# LOAD VALIDATED INPUT DATASETS
# Notebook: 05_Data_Integration_v2.ipynb
# ============================================================

# ------------------------------------------------------------
# Load the WAHIS analytical table.
# This is the disease surveillance dataset produced by
# Notebook 03 — WAHIS Analytical Table.
# ------------------------------------------------------------
wahis = pd.read_csv(
    WAHIS_FILE
)


# ------------------------------------------------------------
# Load the validated administrative boundary lookup.
# This was produced by the administrative boundary validation
# notebook and contains the explicit geographic reconciliation.
# ------------------------------------------------------------
boundary_lookup = pd.read_csv(
    BOUNDARY_FILE
)


# ------------------------------------------------------------
# Load the semester-level climate dataset.
# This dataset is already aggregated to ADM1 + year + semester,
# which matches the temporal structure required for integration.
# ------------------------------------------------------------
climate = pd.read_csv(
    CLIMATE_FILE
)


# ------------------------------------------------------------
# Display dataset dimensions.
# ------------------------------------------------------------
print("=" * 70)
print("INPUT DATASETS LOADED")
print("=" * 70)

print(
    f"\nWAHIS analytical table: "
    f"{wahis.shape[0]:,} rows × {wahis.shape[1]} columns"
)

print(
    f"Boundary lookup: "
    f"{boundary_lookup.shape[0]:,} rows × "
    f"{boundary_lookup.shape[1]} columns"
)

print(
    f"Semester climate dataset: "
    f"{climate.shape[0]:,} rows × "
    f"{climate.shape[1]} columns"
)


# ------------------------------------------------------------
# Display the column names of each dataset.
# ------------------------------------------------------------
print("\nWAHIS columns:")
print(wahis.columns.tolist())

print("\nBoundary lookup columns:")
print(boundary_lookup.columns.tolist())

print("\nClimate columns:")
print(climate.columns.tolist())


# ------------------------------------------------------------
# Display a small preview of each dataset.
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("WAHIS PREVIEW")
print("=" * 70)

display(wahis.head())


print("\n" + "=" * 70)
print("BOUNDARY LOOKUP PREVIEW")
print("=" * 70)

display(boundary_lookup.head())


print("\n" + "=" * 70)
print("CLIMATE PREVIEW")
print("=" * 70)

display(climate.head())

INPUT DATASETS LOADED

WAHIS analytical table: 1,180 rows × 22 columns
Boundary lookup: 39 rows × 7 columns
Semester climate dataset: 518 rows × 13 columns

WAHIS columns:
['analytical_record_id', 'year', 'semester', 'country', 'world_region', 'administrative_division', 'disease', 'animal_category', 'species', 'new_outbreaks', 'outbreak_count_observed', 'outbreak_observation_status', 'serotype_subtype_genotype', 'event_id', 'outbreak_id', 'susceptible', 'measuring_units', 'cases', 'killed_and_disposed_of', 'slaughtered', 'deaths', 'vaccinated']

Boundary lookup columns:
['administrative_division', 'admin_division_standard', 'validated_boundary_name', 'geographic_level', 'boundary_match', 'geographic_match_status', 'validation_note']

Climate columns:
['administrative_division', 'year', 'semester', 'period', 'months_observed', 'temperature_c', 'dewpoint_c', 'semester_total_precipitation_mm', 'semester_mean_wind_component_magnitude_ms', 'minimum_temperature_c_spatial_coverage', 'minimum_

,analytical_record_id,year,semester,country,world_region,administrative_division,disease,animal_category,species,new_outbreaks,...,serotype_subtype_genotype,event_id,outbreak_id,susceptible,measuring_units,cases,killed_and_disposed_of,slaughtered,deaths,vaccinated
0,1,2005,Jul-Dec 2005,Nigeria,Africa,Nigeria,Fowl pox (-2005),Domestic,Birds,NaN,...,NaN,NaN,NaN,90.0,Animal,60.0,0.0,0.0,0.0,0.0
1,2,2005,Jul-Dec 2005,Nigeria,Africa,Nigeria,Fowl typhoid,Domestic,Birds,NaN,...,NaN,NaN,NaN,1000.0,Animal,800.0,0.0,0.0,25.0,0.0
2,3,2005,Jul-Dec 2005,Nigeria,Africa,Nigeria,Infectious bursal disease (Gumboro disease),Domestic,Birds,NaN,...,NaN,NaN,NaN,200.0,Animal,10.0,0.0,0.0,0.0,0.0
3,4,2005,Jul-Dec 2005,Nigeria,Africa,Nigeria,Newcastle disease virus (Inf. with),Domestic,Birds,NaN,...,NaN,NaN,NaN,22100.0,Animal,3527.0,0.0,800.0,2076.0,0.0
4,5,2005,Jan-Jun 2005,Nigeria,Africa,Nigeria,Newcastle disease virus (Inf. with),Domestic,Birds,NaN,...,NaN,NaN,NaN,8100.0,Animal,3430.0,0.0,0.0,912.0,13000.0



BOUNDARY LOOKUP PREVIEW


,administrative_division,admin_division_standard,validated_boundary_name,geographic_level,boundary_match,geographic_match_status,validation_note
0,Abia,abia,Abia,ADM1,True,Matched,Standard ADM1 geographic match
1,Adamawa,adamawa,Adamawa,ADM1,True,Matched,Standard ADM1 geographic match
2,Akwa Ibom,akwa ibom,Akwa Ibom,ADM1,True,Matched,Standard ADM1 geographic match
3,Anambra,anambra,Anambra,ADM1,True,Matched,Standard ADM1 geographic match
4,Bauchi,bauchi,Bauchi,ADM1,True,Matched,Standard ADM1 geographic match



CLIMATE PREVIEW


,administrative_division,year,semester,period,months_observed,temperature_c,dewpoint_c,semester_total_precipitation_mm,semester_mean_wind_component_magnitude_ms,minimum_temperature_c_spatial_coverage,minimum_dewpoint_c_spatial_coverage,minimum_precipitation_monthly_total_mm_spatial_coverage,minimum_wind_speed_from_mean_components_ms_spatial_coverage
0,Abia,2019,Jan-Jun,Jan-Jun 2019,6,27.031811,23.073640,986.154888,1.137628,1.0,1.0,1.0,1.0
1,Abia,2019,Jul-Dec,Jul-Dec 2019,6,25.355141,22.622289,1328.148237,0.976967,1.0,1.0,1.0,1.0
2,Abia,2020,Jan-Jun,Jan-Jun 2020,6,27.743014,21.519952,877.354140,0.975223,1.0,1.0,1.0,1.0
3,Abia,2020,Jul-Dec,Jul-Dec 2020,6,25.599192,22.463069,1091.109333,1.169023,1.0,1.0,1.0,1.0
4,Abia,2021,Jan-Jun,Jan-Jun 2021,6,27.652334,22.147942,599.798555,1.176361,1.0,1.0,1.0,1.0


In [5]:
# ============================================================
# STRUCTURAL AND DATA-QUALITY VALIDATION
# Notebook: 05_Data_Integration_v2.ipynb
# ============================================================

print("=" * 70)
print("STRUCTURAL AND DATA-QUALITY VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. Validate WAHIS analytical record IDs.
# ------------------------------------------------------------
print("\n1. WAHIS ANALYTICAL RECORD ID")
print("-" * 50)

wahis_id_missing = wahis["analytical_record_id"].isna().sum()
wahis_id_duplicates = wahis["analytical_record_id"].duplicated().sum()

print(f"Missing analytical record IDs: {wahis_id_missing:,}")
print(f"Duplicate analytical record IDs: {wahis_id_duplicates:,}")


# ------------------------------------------------------------
# 2. Validate WAHIS country.
# ------------------------------------------------------------
print("\n2. WAHIS COUNTRY")
print("-" * 50)

print("Country values:")
print(wahis["country"].value_counts(dropna=False))


# ------------------------------------------------------------
# 3. Validate WAHIS geographic coverage.
# ------------------------------------------------------------
print("\n3. WAHIS GEOGRAPHIC COVERAGE")
print("-" * 50)

print(
    f"Unique administrative values: "
    f"{wahis['administrative_division'].nunique(dropna=True):,}"
)

print("\nAdministrative values:")
print(
    sorted(
        wahis["administrative_division"]
        .dropna()
        .astype(str)
        .unique()
    )
)


# ------------------------------------------------------------
# 4. Validate WAHIS temporal coverage.
# ------------------------------------------------------------
print("\n4. WAHIS TEMPORAL COVERAGE")
print("-" * 50)

print(
    f"Minimum year: "
    f"{wahis['year'].min()}"
)

print(
    f"Maximum year: "
    f"{wahis['year'].max()}"
)

print("\nSemester values:")
print(
    sorted(
        wahis["semester"]
        .dropna()
        .astype(str)
        .unique()
    )
)


# ------------------------------------------------------------
# 5. Validate climate temporal coverage.
# ------------------------------------------------------------
print("\n5. CLIMATE TEMPORAL COVERAGE")
print("-" * 50)

print(
    f"Minimum year: "
    f"{climate['year'].min()}"
)

print(
    f"Maximum year: "
    f"{climate['year'].max()}"
)

print("\nClimate semester values:")
print(
    sorted(
        climate["semester"]
        .dropna()
        .astype(str)
        .unique()
    )
)


# ------------------------------------------------------------
# 6. Validate climate spatial coverage.
# ------------------------------------------------------------
print("\n6. CLIMATE GEOGRAPHIC COVERAGE")
print("-" * 50)

print(
    f"Unique climate administrative divisions: "
    f"{climate['administrative_division'].nunique(dropna=True):,}"
)

print("\nClimate administrative divisions:")
print(
    sorted(
        climate["administrative_division"]
        .dropna()
        .astype(str)
        .unique()
    )
)


# ------------------------------------------------------------
# 7. Check climate spatial-temporal duplicates.
#
# There should be at most one climate observation for each:
#
#     administrative_division + year + semester
#
# This is critical because the climate dataset will later be
# joined to multiple disease-level WAHIS records.
# ------------------------------------------------------------
print("\n7. CLIMATE SPATIAL-TEMPORAL GRAIN")
print("-" * 50)

climate_key_columns = [
    "administrative_division",
    "year",
    "semester"
]

climate_duplicate_keys = (
    climate
    .duplicated(
        subset=climate_key_columns,
        keep=False
    )
    .sum()
)

climate_unique_keys = (
    climate[
        climate_key_columns
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    f"Climate rows: "
    f"{len(climate):,}"
)

print(
    f"Unique ADM1-Year-Semester keys: "
    f"{climate_unique_keys:,}"
)

print(
    f"Rows belonging to duplicated "
    f"ADM1-Year-Semester keys: "
    f"{climate_duplicate_keys:,}"
)


# ------------------------------------------------------------
# 8. Validate boundary lookup.
# ------------------------------------------------------------
print("\n8. ADMINISTRATIVE BOUNDARY LOOKUP")
print("-" * 50)

print(
    f"Boundary lookup rows: "
    f"{len(boundary_lookup):,}"
)

print(
    f"Unique WAHIS administrative divisions: "
    f"{boundary_lookup['administrative_division'].nunique():,}"
)

print("\nGeographic levels:")
print(
    boundary_lookup["geographic_level"]
    .value_counts(dropna=False)
)

print("\nGeographic match status:")
print(
    boundary_lookup["geographic_match_status"]
    .value_counts(dropna=False)
)

print("\nBoundary matches:")
print(
    boundary_lookup["boundary_match"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 9. Validate the WAHIS outbreak observation status.
# ------------------------------------------------------------
print("\n9. OUTBREAK OBSERVATION STATUS")
print("-" * 50)

print(
    wahis["outbreak_observation_status"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 10. Final structural validation summary.
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("STRUCTURAL VALIDATION SUMMARY")
print("=" * 70)

validation_checks = {
    "WAHIS record IDs unique":
        wahis_id_duplicates == 0,

    "Climate ADM1-Year-Semester grain unique":
        climate_duplicate_keys == 0,

    "Boundary lookup contains 39 WAHIS geographic values":
        len(boundary_lookup) == 39,

    "WAHIS contains Nigeria records":
        wahis["country"].eq("Nigeria").all(),

    "Boundary lookup has no failed matches":
        not boundary_lookup["geographic_match_status"]
        .eq("Unmatched")
        .any(),
}

for check, passed in validation_checks.items():
    status = "PASSED" if passed else "FAILED"
    print(f"{status}: {check}")

if not all(validation_checks.values()):
    raise ValueError(
        "One or more structural validation checks failed. "
        "Review the output before continuing."
    )

print("\nStructural validation completed successfully.")

STRUCTURAL AND DATA-QUALITY VALIDATION

1. WAHIS ANALYTICAL RECORD ID
--------------------------------------------------
Missing analytical record IDs: 0
Duplicate analytical record IDs: 0

2. WAHIS COUNTRY
--------------------------------------------------
Country values:
country
Nigeria    1180
Name: count, dtype: int64

3. WAHIS GEOGRAPHIC COVERAGE
--------------------------------------------------
Unique administrative values: 39

Administrative values:
['Abia', 'Adamawa', 'Akwa Ibom', 'Anambra', 'Bauchi', 'Bayelsa', 'Benue', 'Borno', 'Cross River', 'Delta', 'Ebonyi', 'Edo', 'Ekiti', 'Enugu', 'Federal Capital Territory', 'Gombe', 'Imo', 'Jigawa', 'Kaduna', 'Kano', 'Katsina', 'Kebbi', 'Kogi', 'Kwara', 'Lagos', 'Nasarawa', 'Nassarawa', 'Niger', 'Nigeria', 'Ogun', 'Ondo', 'Osun', 'Oyo', 'Plateau', 'Rivers', 'Sokoto', 'Taraba', 'Yobe', 'Zamfara']

4. WAHIS TEMPORAL COVERAGE
--------------------------------------------------
Minimum year: 2005
Maximum year: 2025

Semester values:
['Jan-

In [6]:
# ============================================================
# INTEGRATION KEY STANDARDISATION AND VALIDATION
# Notebook: 05_Data_Integration_v2.ipynb
# ============================================================

print("=" * 70)
print("INTEGRATION KEY STANDARDISATION AND VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Create a standardised WAHIS semester key
# ------------------------------------------------------------
# WAHIS stores semester values as:
#     Jan-Jun 2019
#     Jul-Dec 2019
#
# Climate data stores semester values as:
#     Jan-Jun
#     Jul-Dec
#
# We therefore retain the WAHIS year separately and extract
# only the semester component for the integration key.

wahis["semester_key"] = (
    wahis["semester"]
    .astype(str)
    .str.extract(r"^(Jan-Jun|Jul-Dec)", expand=False)
)

# ------------------------------------------------------------
# 2. Attach the validated ADM1 boundary name to WAHIS
# ------------------------------------------------------------
# The boundary lookup contains the validated geographic name
# that corresponds to the climate dataset's ADM1 naming scheme.
#
# This is an explicit mapping based on the previously validated
# administrative-boundary notebook. No fuzzy matching is used.

wahis = wahis.merge(
    boundary_lookup[
        [
            "administrative_division",
            "validated_boundary_name",
            "geographic_level",
            "geographic_match_status"
        ]
    ],
    on="administrative_division",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 3. Create the WAHIS ADM1 integration key
# ------------------------------------------------------------
# Country-level records such as "Nigeria" do not have an ADM1
# boundary and therefore will intentionally have a missing
# validated ADM1 name.

wahis["adm1_key"] = wahis["validated_boundary_name"]

# ------------------------------------------------------------
# 4. Standardise the climate semester key
# ------------------------------------------------------------
# The climate dataset already uses the required semester format,
# but we standardise it explicitly for consistency.

climate["semester_key"] = (
    climate["semester"]
    .astype(str)
    .str.strip()
)

# The climate administrative division is already represented
# using the validated boundary naming convention.

climate["adm1_key"] = (
    climate["administrative_division"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# 5. Standardise year types
# ------------------------------------------------------------
# Both datasets must use the same data type for year before
# constructing the spatial-temporal integration key.

wahis["year_key"] = pd.to_numeric(
    wahis["year"],
    errors="coerce"
).astype("Int64")

climate["year_key"] = pd.to_numeric(
    climate["year"],
    errors="coerce"
).astype("Int64")

# ------------------------------------------------------------
# 6. Validate the generated WAHIS keys
# ------------------------------------------------------------
print("\n1. WAHIS INTEGRATION KEYS")
print("-" * 50)

print("Semester key values:")
print(sorted(wahis["semester_key"].dropna().unique()))

print("\nADM1 key missing values:")
print(wahis["adm1_key"].isna().sum())

print("\nYear key missing values:")
print(wahis["year_key"].isna().sum())

print("\nSemester key missing values:")
print(wahis["semester_key"].isna().sum())

# ------------------------------------------------------------
# 7. Validate the generated climate keys
# ------------------------------------------------------------
print("\n2. CLIMATE INTEGRATION KEYS")
print("-" * 50)

print("Semester key values:")
print(sorted(climate["semester_key"].dropna().unique()))

print("\nADM1 key missing values:")
print(climate["adm1_key"].isna().sum())

print("\nYear key missing values:")
print(climate["year_key"].isna().sum())

print("\nSemester key missing values:")
print(climate["semester_key"].isna().sum())

# ------------------------------------------------------------
# 8. Compare the three integration dimensions
# ------------------------------------------------------------
print("\n3. INTEGRATION DIMENSION OVERLAP")
print("-" * 50)

wahis_adm1 = set(wahis["adm1_key"].dropna().unique())
climate_adm1 = set(climate["adm1_key"].dropna().unique())

wahis_years = set(wahis["year_key"].dropna().astype(int).unique())
climate_years = set(climate["year_key"].dropna().astype(int).unique())

wahis_semesters = set(wahis["semester_key"].dropna().unique())
climate_semesters = set(climate["semester_key"].dropna().unique())

print(f"Common ADM1 values: {len(wahis_adm1 & climate_adm1):,}")
print(f"WAHIS ADM1 values: {len(wahis_adm1):,}")
print(f"Climate ADM1 values: {len(climate_adm1):,}")

print(f"\nCommon years: {len(wahis_years & climate_years):,}")
print(f"WAHIS years: {min(wahis_years)}–{max(wahis_years)}")
print(f"Climate years: {min(climate_years)}–{max(climate_years)}")

print(f"\nCommon semesters: {len(wahis_semesters & climate_semesters):,}")
print(f"WAHIS semesters: {sorted(wahis_semesters)}")
print(f"Climate semesters: {sorted(climate_semesters)}")

# ------------------------------------------------------------
# 9. Construct the complete integration key
# ------------------------------------------------------------
# The combined key represents the spatial-temporal grain at
# which climate observations are available.

wahis["integration_key"] = (
    wahis["adm1_key"].astype("string")
    + "_"
    + wahis["year_key"].astype("string")
    + "_"
    + wahis["semester_key"].astype("string")
)

climate["integration_key"] = (
    climate["adm1_key"].astype("string")
    + "_"
    + climate["year_key"].astype("string")
    + "_"
    + climate["semester_key"].astype("string")
)

# ------------------------------------------------------------
# 10. Compare complete integration keys
# ------------------------------------------------------------
print("\n4. COMPLETE INTEGRATION KEY OVERLAP")
print("-" * 50)

wahis_keys = set(
    wahis.loc[
        wahis["adm1_key"].notna()
        & wahis["year_key"].notna()
        & wahis["semester_key"].notna(),
        "integration_key"
    ]
)

climate_keys = set(
    climate.loc[
        climate["adm1_key"].notna()
        & climate["year_key"].notna()
        & climate["semester_key"].notna(),
        "integration_key"
    ]
)

common_keys = wahis_keys & climate_keys

print(f"Complete WAHIS ADM1-Year-Semester keys: {len(wahis_keys):,}")
print(f"Complete climate ADM1-Year-Semester keys: {len(climate_keys):,}")
print(f"Common ADM1-Year-Semester keys: {len(common_keys):,}")

if len(climate_keys) > 0:
    key_coverage = len(common_keys) / len(climate_keys) * 100
    print(f"Climate key coverage by WAHIS: {key_coverage:.2f}%")

# ------------------------------------------------------------
# 11. Validation checks
# ------------------------------------------------------------
print("\n5. KEY VALIDATION")
print("-" * 50)

key_checks = {
    "WAHIS semester keys are valid":
        wahis["semester_key"].dropna().isin(["Jan-Jun", "Jul-Dec"]).all(),

    "Climate semester keys are valid":
        climate["semester_key"].dropna().isin(["Jan-Jun", "Jul-Dec"]).all(),

    "WAHIS ADM1 keys are present where geographic level is ADM1":
        wahis.loc[
            wahis["geographic_level"].eq("ADM1"),
            "adm1_key"
        ].notna().all(),

    "Climate integration keys are unique":
        climate["integration_key"].duplicated().sum() == 0,

    "Common integration keys exist":
        len(common_keys) > 0,
}

for check, passed in key_checks.items():
    status = "PASSED" if passed else "FAILED"
    print(f"{status}: {check}")

if not all(key_checks.values()):
    raise ValueError(
        "One or more integration-key validation checks failed. "
        "Review the output before continuing."
    )

print("\n" + "=" * 70)
print("INTEGRATION KEY VALIDATION COMPLETED SUCCESSFULLY")
print("=" * 70)

INTEGRATION KEY STANDARDISATION AND VALIDATION

1. WAHIS INTEGRATION KEYS
--------------------------------------------------
Semester key values:
['Jan-Jun', 'Jul-Dec']

ADM1 key missing values:
1

Year key missing values:
0

Semester key missing values:
0

2. CLIMATE INTEGRATION KEYS
--------------------------------------------------
Semester key values:
['Jan-Jun', 'Jul-Dec']

ADM1 key missing values:
0

Year key missing values:
0

Semester key missing values:
0

3. INTEGRATION DIMENSION OVERLAP
--------------------------------------------------
Common ADM1 values: 37
WAHIS ADM1 values: 38
Climate ADM1 values: 37

Common years: 7
WAHIS years: 2005–2025
Climate years: 2019–2025

Common semesters: 2
WAHIS semesters: ['Jan-Jun', 'Jul-Dec']
Climate semesters: ['Jan-Jun', 'Jul-Dec']

4. COMPLETE INTEGRATION KEY OVERLAP
--------------------------------------------------
Complete WAHIS ADM1-Year-Semester keys: 416
Complete climate ADM1-Year-Semester keys: 518
Common ADM1-Year-Semester keys:

In [7]:
# ============================================================
# SPATIAL-TEMPORAL CLIMATE INTEGRATION
# Notebook: 05_Data_Integration_v2.ipynb
# ============================================================

print("=" * 70)
print("SPATIAL-TEMPORAL CLIMATE INTEGRATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Define the climate variables to attach to WAHIS
# ------------------------------------------------------------
# We only bring climate variables and the integration key
# information into the WAHIS analytical table.
#
# The climate dataset has one unique record per:
#     ADM1 + Year + Semester
#
# Therefore, the merge should be many-to-one:
#
#     many WAHIS disease records
#              ↓
#     one climate observation
#
# for each ADM1-Year-Semester combination.

climate_columns = [
    "integration_key",
    "adm1_key",
    "year_key",
    "semester_key",
    "months_observed",
    "temperature_c",
    "dewpoint_c",
    "semester_total_precipitation_mm",
    "semester_mean_wind_component_magnitude_ms",
    "minimum_temperature_c_spatial_coverage",
    "minimum_dewpoint_c_spatial_coverage",
    "minimum_precipitation_monthly_total_mm_spatial_coverage",
    "minimum_wind_speed_from_mean_components_ms_spatial_coverage"
]

climate_for_merge = climate[climate_columns].copy()

# ------------------------------------------------------------
# 2. Confirm the climate merge key is unique
# ------------------------------------------------------------
# This is a critical safeguard.
#
# If the climate key were duplicated, a many-to-one merge would
# incorrectly multiply WAHIS disease records.

duplicate_climate_keys = climate_for_merge[
    "integration_key"
].duplicated().sum()

print("\n1. CLIMATE MERGE-KEY VALIDATION")
print("-" * 50)
print(f"Duplicate climate integration keys: {duplicate_climate_keys:,}")

if duplicate_climate_keys != 0:
    raise ValueError(
        "Climate integration keys are not unique. "
        "The merge cannot safely proceed."
    )

# ------------------------------------------------------------
# 3. Record the original WAHIS row count
# ------------------------------------------------------------
# This provides a baseline for verifying that the integration
# does not accidentally create or remove WAHIS records.

original_wahis_rows = len(wahis)

print(f"Original WAHIS rows: {original_wahis_rows:,}")

# ------------------------------------------------------------
# 4. Perform the many-to-one merge
# ------------------------------------------------------------
# A LEFT JOIN is used deliberately.
#
# This preserves every WAHIS record, including:
# - records outside the climate period
# - records with unavailable outbreak counts
# - the country-level Nigeria record
# - ADM1-semester combinations for which climate data are absent
#
# Missing climate values after the merge mean that no matching
# climate observation was available for that spatial-temporal key.
#
# They must NOT be converted to zero.

integrated = wahis.merge(
    climate_for_merge,
    on="integration_key",
    how="left",
    validate="many_to_one",
    suffixes=("", "_climate")
)

# ------------------------------------------------------------
# 5. Create an explicit climate match status
# ------------------------------------------------------------
# A climate match exists when the climate integration key found
# a corresponding climate observation.
#
# We use temperature as an availability indicator because it is
# one of the primary climate variables brought from the climate
# dataset. The integration key itself is retained for traceability.

integrated["climate_match_status"] = np.where(
    integrated["temperature_c"].notna(),
    "Matched",
    "Unavailable"
)

# ------------------------------------------------------------
# 6. Validate row preservation
# ------------------------------------------------------------
integrated_rows = len(integrated)

print("\n2. ROW-PRESERVATION VALIDATION")
print("-" * 50)
print(f"Original WAHIS rows: {original_wahis_rows:,}")
print(f"Integrated rows: {integrated_rows:,}")

if integrated_rows != original_wahis_rows:
    raise ValueError(
        "Row-preservation check failed. "
        "The climate merge changed the number of WAHIS records."
    )

print("Row-preservation check: PASSED")

# ------------------------------------------------------------
# 7. Validate integrated spatial-temporal grain
# ------------------------------------------------------------
# WAHIS contains multiple disease records for the same
# ADM1-Year-Semester combination, so repeated integration keys
# are expected in the integrated dataset.
#
# What must remain unique is the climate source key.

integrated_duplicate_keys = integrated[
    "integration_key"
].duplicated().sum()

print("\n3. INTEGRATED KEY STRUCTURE")
print("-" * 50)
print(
    f"Integrated rows with repeated ADM1-Year-Semester keys: "
    f"{integrated_duplicate_keys:,}"
)

print(
    "Repeated keys are expected because multiple WAHIS disease "
    "records can share the same climate observation."
)

# ------------------------------------------------------------
# 8. Calculate climate match coverage
# ------------------------------------------------------------
match_counts = (
    integrated["climate_match_status"]
    .value_counts(dropna=False)
    .rename_axis("climate_match_status")
    .reset_index(name="records")
)

print("\n4. CLIMATE MATCH COVERAGE")
print("-" * 50)
print(match_counts.to_string(index=False))

matched_records = (
    integrated["climate_match_status"]
    .eq("Matched")
    .sum()
)

climate_match_rate = (
    matched_records / integrated_rows * 100
)

print(f"\nClimate match rate: {climate_match_rate:.2f}%")

# ------------------------------------------------------------
# 9. Examine climate coverage by year
# ------------------------------------------------------------
# This helps distinguish expected temporal limitations from
# geographic integration problems.

print("\n5. CLIMATE COVERAGE BY YEAR")
print("-" * 50)

coverage_by_year = (
    integrated.groupby("year", dropna=False)
    .agg(
        total_records=("analytical_record_id", "size"),
        climate_matched_records=(
            "climate_match_status",
            lambda x: (x == "Matched").sum()
        )
    )
    .reset_index()
)

coverage_by_year["climate_match_rate_pct"] = (
    coverage_by_year["climate_matched_records"]
    / coverage_by_year["total_records"]
    * 100
)

print(coverage_by_year.to_string(index=False))

# ------------------------------------------------------------
# 10. Examine climate coverage by semester
# ------------------------------------------------------------
print("\n6. CLIMATE COVERAGE BY SEMESTER")
print("-" * 50)

coverage_by_semester = (
    integrated.groupby("semester_key", dropna=False)
    .agg(
        total_records=("analytical_record_id", "size"),
        climate_matched_records=(
            "climate_match_status",
            lambda x: (x == "Matched").sum()
        )
    )
    .reset_index()
)

coverage_by_semester["climate_match_rate_pct"] = (
    coverage_by_semester["climate_matched_records"]
    / coverage_by_semester["total_records"]
    * 100
)

print(coverage_by_semester.to_string(index=False))

# ------------------------------------------------------------
# 11. Examine coverage by geographic level
# ------------------------------------------------------------
# The country-level Nigeria record is intentionally not matched
# to ADM1 climate data.

print("\n7. CLIMATE COVERAGE BY GEOGRAPHIC LEVEL")
print("-" * 50)

coverage_by_geography = (
    integrated.groupby(
        ["geographic_level", "climate_match_status"],
        dropna=False
    )
    .size()
    .reset_index(name="records")
)

print(coverage_by_geography.to_string(index=False))

# ------------------------------------------------------------
# 12. Check outbreak observation status after integration
# ------------------------------------------------------------
# This confirms that climate integration has not changed the
# surveillance observation-status logic.

print("\n8. OUTBREAK OBSERVATION STATUS")
print("-" * 50)

print(
    integrated["outbreak_observation_status"]
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------
# 13. Final integration validation
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("INTEGRATION VALIDATION SUMMARY")
print("=" * 70)

integration_checks = {
    "Integrated row count equals original WAHIS row count":
        len(integrated) == original_wahis_rows,

    "Climate merge key was unique":
        duplicate_climate_keys == 0,

    "Climate matches were successfully identified":
        matched_records > 0,

    "Country-level Nigeria record remains in dataset":
        (
            integrated.loc[
                integrated["geographic_level"].eq("Country"),
                "administrative_division"
            ].eq("Nigeria").all()
        ),

    "Unavailable outbreak observations remain unavailable":
        (
            integrated.loc[
                integrated["outbreak_count_observed"].eq(False),
                "outbreak_observation_status"
            ].eq("Unavailable").all()
        )
}

for check, passed in integration_checks.items():
    status = "PASSED" if passed else "FAILED"
    print(f"{status}: {check}")

if not all(integration_checks.values()):
    raise ValueError(
        "One or more climate integration validation checks failed. "
        "Review the output before continuing."
    )

print("\n" + "=" * 70)
print("SPATIAL-TEMPORAL CLIMATE INTEGRATION COMPLETED")
print("=" * 70)

SPATIAL-TEMPORAL CLIMATE INTEGRATION

1. CLIMATE MERGE-KEY VALIDATION
--------------------------------------------------
Duplicate climate integration keys: 0
Original WAHIS rows: 1,180

2. ROW-PRESERVATION VALIDATION
--------------------------------------------------
Original WAHIS rows: 1,180
Integrated rows: 1,180
Row-preservation check: PASSED

3. INTEGRATED KEY STRUCTURE
--------------------------------------------------
Integrated rows with repeated ADM1-Year-Semester keys: 763
Repeated keys are expected because multiple WAHIS disease records can share the same climate observation.

4. CLIMATE MATCH COVERAGE
--------------------------------------------------
climate_match_status  records
             Matched      984
         Unavailable      196

Climate match rate: 83.39%

5. CLIMATE COVERAGE BY YEAR
--------------------------------------------------
 year  total_records  climate_matched_records  climate_match_rate_pct
 2005              5                        0              

In [8]:
# ============================================================
# FINAL INTEGRATED DATASET VALIDATION
# Notebook: 05_Data_Integration_v2.ipynb
# ============================================================

print("=" * 70)
print("FINAL INTEGRATED DATASET VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Check the integrated dataset structure
# ------------------------------------------------------------
print("\n1. INTEGRATED DATASET STRUCTURE")
print("-" * 50)

print(f"Rows: {len(integrated):,}")
print(f"Columns: {len(integrated.columns):,}")

print("\nIntegrated columns:")
print(integrated.columns.tolist())

# ------------------------------------------------------------
# 2. Check analytical record identity
# ------------------------------------------------------------
# Every original WAHIS analytical record must remain present
# exactly once after integration.

print("\n2. ANALYTICAL RECORD ID VALIDATION")
print("-" * 50)

missing_record_ids = integrated["analytical_record_id"].isna().sum()
duplicate_record_ids = integrated["analytical_record_id"].duplicated().sum()

print(f"Missing analytical record IDs: {missing_record_ids:,}")
print(f"Duplicate analytical record IDs: {duplicate_record_ids:,}")

# ------------------------------------------------------------
# 3. Check climate match consistency
# ------------------------------------------------------------
# A matched record should have a corresponding integration key
# and at least one climate observation.
#
# An unavailable record should not have climate values attached.

print("\n3. CLIMATE MATCH CONSISTENCY")
print("-" * 50)

matched_without_temperature = integrated.loc[
    integrated["climate_match_status"].eq("Matched"),
    "temperature_c"
].isna().sum()

unavailable_with_temperature = integrated.loc[
    integrated["climate_match_status"].eq("Unavailable"),
    "temperature_c"
].notna().sum()

print(
    "Matched records without temperature: "
    f"{matched_without_temperature:,}"
)

print(
    "Unavailable records with temperature: "
    f"{unavailable_with_temperature:,}"
)

# ------------------------------------------------------------
# 4. Check climate coverage period
# ------------------------------------------------------------
print("\n4. CLIMATE TEMPORAL COVERAGE IN INTEGRATED DATA")
print("-" * 50)

matched_years = sorted(
    integrated.loc[
        integrated["climate_match_status"].eq("Matched"),
        "year"
    ].dropna().unique()
)

print(f"Years with matched climate data: {matched_years}")

if matched_years:
    print(f"First matched climate year: {min(matched_years)}")
    print(f"Last matched climate year: {max(matched_years)}")

# ------------------------------------------------------------
# 5. Check ADM1 climate coverage
# ------------------------------------------------------------
# This identifies whether any ADM1 names unexpectedly failed
# to match after geographic validation.

print("\n5. ADM1 CLIMATE COVERAGE")
print("-" * 50)

adm1_coverage = (
    integrated.loc[
        integrated["geographic_level"].eq("ADM1")
    ]
    .groupby(
        "climate_match_status"
    )
    .size()
    .reset_index(name="records")
)

print(adm1_coverage.to_string(index=False))

# ------------------------------------------------------------
# 6. Identify ADM1-semester combinations without climate
# ------------------------------------------------------------
# These records are retained. We investigate them rather than
# treating them as errors.

print("\n6. UNMATCHED ADM1-YEAR-SEMESTER COMBINATIONS")
print("-" * 50)

unmatched_keys = (
    integrated.loc[
        integrated["climate_match_status"].eq("Unavailable")
        & integrated["geographic_level"].eq("ADM1")
    ][
        [
            "adm1_key",
            "year_key",
            "semester_key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["year_key", "semester_key", "adm1_key"]
    )
)

print(
    f"Unique unmatched ADM1-Year-Semester combinations: "
    f"{len(unmatched_keys):,}"
)

print("\nSample of unmatched combinations:")
print(unmatched_keys.head(20).to_string(index=False))

# ------------------------------------------------------------
# 7. Check outbreak surveillance variables
# ------------------------------------------------------------
# Climate integration must not alter the original WAHIS
# surveillance variables or their observation-status logic.

print("\n7. OUTBREAK SURVEILLANCE VALIDATION")
print("-" * 50)

print("Outbreak observation status:")
print(
    integrated["outbreak_observation_status"]
    .value_counts(dropna=False)
    .to_string()
)

observed_with_missing_count = integrated.loc[
    integrated["outbreak_observation_status"].eq("Observed"),
    "new_outbreaks"
].isna().sum()

unavailable_with_count = integrated.loc[
    integrated["outbreak_observation_status"].eq("Unavailable"),
    "new_outbreaks"
].notna().sum()

print(
    f"\nObserved records with missing new_outbreaks: "
    f"{observed_with_missing_count:,}"
)

print(
    f"Unavailable records with a new_outbreaks value: "
    f"{unavailable_with_count:,}"
)

# ------------------------------------------------------------
# 8. Check for accidental duplicate complete records
# ------------------------------------------------------------
# Multiple disease records can legitimately share the same
# ADM1-Year-Semester climate observation, so duplicate
# integration keys are expected.
#
# However, exact duplicate integrated records should not occur.

duplicate_complete_rows = integrated.duplicated().sum()

print("\n8. COMPLETE-ROW DUPLICATE CHECK")
print("-" * 50)
print(f"Duplicate complete integrated rows: {duplicate_complete_rows:,}")

# ------------------------------------------------------------
# 9. Check climate variable availability
# ------------------------------------------------------------
print("\n9. CLIMATE VARIABLE AVAILABILITY")
print("-" * 50)

climate_variables = [
    "temperature_c",
    "dewpoint_c",
    "semester_total_precipitation_mm",
    "semester_mean_wind_component_magnitude_ms"
]

for column in climate_variables:
    available = integrated[column].notna().sum()
    availability_rate = available / len(integrated) * 100

    print(
        f"{column}: "
        f"{available:,}/{len(integrated):,} "
        f"({availability_rate:.2f}% available)"
    )

# ------------------------------------------------------------
# 10. Final validation checks
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FINAL VALIDATION SUMMARY")
print("=" * 70)

final_checks = {
    "Integrated dataset contains 1,180 WAHIS records":
        len(integrated) == original_wahis_rows,

    "Analytical record IDs are unique":
        duplicate_record_ids == 0,

    "No analytical record IDs are missing":
        missing_record_ids == 0,

    "Matched climate records have temperature values":
        matched_without_temperature == 0,

    "Unavailable climate records have no temperature values":
        unavailable_with_temperature == 0,

    "No duplicate complete integrated rows":
        duplicate_complete_rows == 0,

    "Observed outbreak records have outbreak counts":
        observed_with_missing_count == 0,

    "Unavailable outbreak records do not contain outbreak counts":
        unavailable_with_count == 0,
}

for check, passed in final_checks.items():
    status = "PASSED" if passed else "FAILED"
    print(f"{status}: {check}")

if not all(final_checks.values()):
    raise ValueError(
        "One or more final integrated-dataset validation checks failed. "
        "Review the output before saving the dataset."
    )

print("\n" + "=" * 70)
print("FINAL INTEGRATED DATASET VALIDATION COMPLETED SUCCESSFULLY")
print("=" * 70)

FINAL INTEGRATED DATASET VALIDATION

1. INTEGRATED DATASET STRUCTURE
--------------------------------------------------
Rows: 1,180
Columns: 42

Integrated columns:
['analytical_record_id', 'year', 'semester', 'country', 'world_region', 'administrative_division', 'disease', 'animal_category', 'species', 'new_outbreaks', 'outbreak_count_observed', 'outbreak_observation_status', 'serotype_subtype_genotype', 'event_id', 'outbreak_id', 'susceptible', 'measuring_units', 'cases', 'killed_and_disposed_of', 'slaughtered', 'deaths', 'vaccinated', 'semester_key', 'validated_boundary_name', 'geographic_level', 'geographic_match_status', 'adm1_key', 'year_key', 'integration_key', 'adm1_key_climate', 'year_key_climate', 'semester_key_climate', 'months_observed', 'temperature_c', 'dewpoint_c', 'semester_total_precipitation_mm', 'semester_mean_wind_component_magnitude_ms', 'minimum_temperature_c_spatial_coverage', 'minimum_dewpoint_c_spatial_coverage', 'minimum_precipitation_monthly_total_mm_spatial_

In [9]:
# ============================================================
# FINALISE AND SAVE INTEGRATED DATASET
# Notebook: 05_Data_Integration_v2.ipynb
# ============================================================

print("=" * 70)
print("FINALISING AND SAVING WAHIS–CLIMATE INTEGRATED DATASET")
print("=" * 70)

# ------------------------------------------------------------
# 1. Remove redundant climate merge-helper columns
# ------------------------------------------------------------
# These columns were created automatically during the merge
# because WAHIS and climate both contained ADM1, year, and
# semester helper fields.
#
# They are not required in the final analytical dataset because
# the primary integration keys are already retained.

redundant_columns = [
    "adm1_key_climate",
    "year_key_climate",
    "semester_key_climate"
]

columns_to_drop = [
    column
    for column in redundant_columns
    if column in integrated.columns
]

integrated = integrated.drop(columns=columns_to_drop)

print("\n1. REDUNDANT COLUMN CLEAN-UP")
print("-" * 50)

if columns_to_drop:
    print("Removed columns:")
    for column in columns_to_drop:
        print(f"  - {column}")
else:
    print("No redundant merge-helper columns found.")

# ------------------------------------------------------------
# 2. Define final column order
# ------------------------------------------------------------
# The final dataset is organised into logical groups:
#
# A. WAHIS identifiers and surveillance information
# B. Geographic information
# C. Integration keys
# D. Climate variables
# E. Climate coverage metadata

preferred_column_order = [
    # WAHIS identifiers and core dimensions
    "analytical_record_id",
    "year",
    "semester",
    "country",
    "world_region",
    "administrative_division",
    "disease",
    "animal_category",
    "species",

    # Outbreak surveillance
    "new_outbreaks",
    "outbreak_count_observed",
    "outbreak_observation_status",
    "serotype_subtype_genotype",
    "event_id",
    "outbreak_id",
    "susceptible",
    "measuring_units",
    "cases",
    "killed_and_disposed_of",
    "slaughtered",
    "deaths",
    "vaccinated",

    # Validated geographic information
    "validated_boundary_name",
    "geographic_level",
    "geographic_match_status",
    "adm1_key",

    # Integration keys
    "year_key",
    "semester_key",
    "integration_key",

    # Climate data
    "months_observed",
    "temperature_c",
    "dewpoint_c",
    "semester_total_precipitation_mm",
    "semester_mean_wind_component_magnitude_ms",

    # Climate spatial coverage indicators
    "minimum_temperature_c_spatial_coverage",
    "minimum_dewpoint_c_spatial_coverage",
    "minimum_precipitation_monthly_total_mm_spatial_coverage",
    "minimum_wind_speed_from_mean_components_ms_spatial_coverage",

    # Climate availability indicator
    "climate_match_status"
]

# Keep only columns that actually exist.
final_columns = [
    column
    for column in preferred_column_order
    if column in integrated.columns
]

integrated = integrated[final_columns]

# ------------------------------------------------------------
# 3. Final dataset integrity checks
# ------------------------------------------------------------
print("\n2. FINAL DATASET STRUCTURE")
print("-" * 50)

print(f"Rows: {len(integrated):,}")
print(f"Columns: {len(integrated.columns):,}")

print("\nFinal columns:")
print(integrated.columns.tolist())

# ------------------------------------------------------------
# 4. Confirm analytical record uniqueness
# ------------------------------------------------------------
final_duplicate_ids = integrated[
    "analytical_record_id"
].duplicated().sum()

final_duplicate_rows = integrated.duplicated().sum()

print("\n3. FINAL UNIQUENESS CHECK")
print("-" * 50)

print(
    f"Duplicate analytical record IDs: "
    f"{final_duplicate_ids:,}"
)

print(
    f"Duplicate complete rows: "
    f"{final_duplicate_rows:,}"
)

# ------------------------------------------------------------
# 5. Confirm climate coverage
# ------------------------------------------------------------
print("\n4. FINAL CLIMATE COVERAGE")
print("-" * 50)

final_climate_counts = (
    integrated["climate_match_status"]
    .value_counts(dropna=False)
)

print(final_climate_counts.to_string())

final_climate_match_rate = (
    integrated["climate_match_status"].eq("Matched").mean() * 100
)

print(
    f"\nFinal climate match rate: "
    f"{final_climate_match_rate:.2f}%"
)

# ------------------------------------------------------------
# 6. Confirm outbreak observation logic
# ------------------------------------------------------------
print("\n5. FINAL OUTBREAK OBSERVATION STATUS")
print("-" * 50)

print(
    integrated["outbreak_observation_status"]
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------
# 7. Save the final integrated dataset
# ------------------------------------------------------------
# This becomes the primary input dataset for Notebook 06 —
# Feature Engineering.

integrated.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n6. OUTPUT FILE")
print("-" * 50)
print(f"Saved to: {OUTPUT_FILE}")
print(f"File exists: {OUTPUT_FILE.exists()}")

# ------------------------------------------------------------
# 8. Reload the saved dataset to verify persistence
# ------------------------------------------------------------
# Reloading the file confirms that the CSV can be read back
# successfully and retains the expected structure.

integrated_saved = pd.read_csv(OUTPUT_FILE)

print("\n7. SAVED FILE VALIDATION")
print("-" * 50)

print(f"Reloaded rows: {len(integrated_saved):,}")
print(f"Reloaded columns: {len(integrated_saved.columns):,}")

saved_structure_matches = (
    integrated_saved.shape == integrated.shape
)

saved_id_duplicates = (
    integrated_saved["analytical_record_id"].duplicated().sum()
)

print(
    f"Saved structure matches in-memory dataset: "
    f"{saved_structure_matches}"
)

print(
    f"Duplicate analytical record IDs after reload: "
    f"{saved_id_duplicates:,}"
)

# ------------------------------------------------------------
# 9. Final save validation
# ------------------------------------------------------------
save_checks = {
    "Final dataset contains 1,180 rows":
        len(integrated) == 1180,

    "Analytical record IDs remain unique":
        final_duplicate_ids == 0,

    "No duplicate complete rows":
        final_duplicate_rows == 0,

    "Final climate match rate is 83.39%":
        round(final_climate_match_rate, 2) == 83.39,

    "Saved file exists":
        OUTPUT_FILE.exists(),

    "Saved file structure matches in-memory dataset":
        saved_structure_matches,

    "Saved analytical record IDs remain unique":
        saved_id_duplicates == 0,
}

print("\n" + "=" * 70)
print("FINAL SAVE VALIDATION")
print("=" * 70)

for check, passed in save_checks.items():
    status = "PASSED" if passed else "FAILED"
    print(f"{status}: {check}")

if not all(save_checks.values()):
    raise ValueError(
        "One or more final save-validation checks failed. "
        "Review the output before proceeding."
    )

print("\n" + "=" * 70)
print("NOTEBOOK 05 DATA INTEGRATION COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nPrimary output:")
print(OUTPUT_FILE)

FINALISING AND SAVING WAHIS–CLIMATE INTEGRATED DATASET

1. REDUNDANT COLUMN CLEAN-UP
--------------------------------------------------
Removed columns:
  - adm1_key_climate
  - year_key_climate
  - semester_key_climate

2. FINAL DATASET STRUCTURE
--------------------------------------------------
Rows: 1,180
Columns: 39

Final columns:
['analytical_record_id', 'year', 'semester', 'country', 'world_region', 'administrative_division', 'disease', 'animal_category', 'species', 'new_outbreaks', 'outbreak_count_observed', 'outbreak_observation_status', 'serotype_subtype_genotype', 'event_id', 'outbreak_id', 'susceptible', 'measuring_units', 'cases', 'killed_and_disposed_of', 'slaughtered', 'deaths', 'vaccinated', 'validated_boundary_name', 'geographic_level', 'geographic_match_status', 'adm1_key', 'year_key', 'semester_key', 'integration_key', 'months_observed', 'temperature_c', 'dewpoint_c', 'semester_total_precipitation_mm', 'semester_mean_wind_component_magnitude_ms', 'minimum_temperatur